# Task 3: Exploratory Data Analysis on IPL Dataset

Dataset: `IPL_Ball_by_Ball_2008_2022.csv`, `IPL_Ball_by_Ball_2022.csv`, and `IPL_Matches_2022.csv`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 1. Basic Data Preprocessing


In [ ]:
ipl_all = pd.read_csv("C:/Users/tasik/Python/datasets/IPL_Ball_by_Ball_2008_2022.csv")
ball = pd.read_csv("C:/Users/tasik/Python/datasets/IPL_Ball_by_Ball_2022.csv")
match = pd.read_csv("C:/Users/tasik/Python/datasets/IPL_Matches_2022.csv")


In [ ]:
print("All ball-by-ball shape:", ipl_all.shape)
print("2022 ball-by-ball shape:", ball.shape)
print("2022 matches shape:", match.shape)


In [ ]:
ball.info()


In [ ]:
ball.sample(2)


## 2. Highest Wicket Taker in a Particular Season

The available match file contains the 2022 season, so the season-wise wicket analysis below is calculated for IPL 2022.


In [ ]:
bowler_wicket_kinds = {
    "bowled", "caught", "caught and bowled", "lbw", "stumped", "hit wicket"
}

season_wickets = ball[ball["kind"].isin(bowler_wicket_kinds)].merge(
    match[["ID", "Season"]], on="ID", how="left"
)

wicket_counts = (
    season_wickets.groupby(["Season", "bowler"])
    .size()
    .rename("wickets")
    .reset_index()
)

top_idx = wicket_counts.groupby("Season")["wickets"].idxmax()
season_wicket_takers = wicket_counts.loc[top_idx].sort_values("Season")

season_wicket_takers

## 3. Player Records in IPL History

The following records are calculated from `IPL_Ball_by_Ball_2008_2022.csv`.


In [ ]:
# 1. Most deliveries faced — a 'wide' isn't a ball faced by the batter
faced_counts = ball.loc[ball['extra_type'] != 'wides', 'batter'].value_counts()
top_faced = faced_counts.idxmax(), faced_counts.max()
top_faced

# 2. Most sixes
six_counts = ball.loc[ball['batsman_run'] == 6, 'batter'].value_counts()
top_sixes = six_counts.idxmax(), six_counts.max()
top_sixes
# 3. Most catches — credit the fielder; for 'caught and bowled' the bowler IS the catcher
caught = ball[ball['kind'].isin(['caught', 'caught and bowled'])]
catcher = caught['fielders_involved'].fillna(caught['bowler'])
catch_counts = catcher.value_counts()
top_catches = catch_counts.idxmax(), catch_counts.max()
top_catches
# 4. Most LBW wickets
lbw_counts = ball.loc[ball['kind'] == 'lbw', 'bowler'].value_counts()
top_lbw = lbw_counts.idxmax(), lbw_counts.max()
top_lbw

## 4. Highest Run Getter in the Season and Chart

The available season mapping is for IPL 2022, so this table and chart show the highest run getter for the 2022 season.


In [ ]:

season_ball = ball.merge(match[["ID", "Season"]], on="ID", how="left")

season_runs = (
    season_ball.groupby(["Season", "batter"])["batsman_run"]
    .sum()
    .rename("runs")
    .reset_index()
)

# nlargest is O(n) vs sort_values + head on the full table
top10 = season_runs.nlargest(10, "runs").sort_values("runs")  # ascending for barh

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10["batter"], top10["runs"], color="#1f77b4")
ax.bar_label(bars, padding=3)
ax.set_xlabel("Runs")
ax.set_title("Top 10 Run Scorers — IPL 2022")
plt.tight_layout()
plt.savefig("top10_runs.png", dpi=150)
plt.show()

## 5. Top 5 Batsmen by Strike Rate in IPL History

Only players who faced at least 50 legal deliveries are considered.


In [ ]:
MIN_BALLS = 60  # ~10 overs faced — filters out small-sample noise

# Runs per batter (single groupby-sum, O(n))
runs = ball.groupby("batter")["batsman_run"].sum().rename("runs")

# Balls faced excludes wides (batter isn't credited a "faced" ball on a wide)
balls_faced = (
    ball.loc[ball["extra_type"] != "wides", "batter"]
    .value_counts()
    .rename("balls")
)

# Single concat instead of two merges
stats = pd.concat([runs, balls_faced], axis=1).dropna()
stats["strike_rate"] = (stats["runs"] / stats["balls"]) * 100

top5_sr = (
    stats[stats["balls"] >= MIN_BALLS]
    .nlargest(5, "strike_rate")
    .reset_index()
)
print(top5_sr)